In [8]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# ==============================================================================
# 1. CONFIGURAÇÃO DO MEDIAPIPE E ARQUIVO DE DADOS
# ==============================================================================
arquivo_csv = "meu_dataset_libras.csv"
colunas = [f"{eixo}_{i}" for i in range(21) for eixo in ['x', 'y', 'z']]

# Se já existir um dataset gravado antes, carrega para você poder adicionar mais sinais
if os.path.exists(arquivo_csv):
    df_existente = pd.read_csv(arquivo_csv)
    dados_coletados = df_existente.drop(columns=['classe']).values.tolist()
    labels_coletados = df_existente['classe'].astype(str).tolist()
    print(f"Dataset anterior encontrado! Carregadas {len(labels_coletados)} amostras.")
else:
    dados_coletados = []
    labels_coletados = []

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("\n" + "="*60)
print(" INSTRUÇÕES DE GRAVAÇÃO PELA WEBCAM:")
print(" 1. Faça o sinal com a mão na frente da câmera.")
print(" 2. Pressione a tecla da letra correspondente (ex: 'a', 'b', 'c'...).")
print("    -> Cada clique grava uma rajada automática de 50 amostras!")
print("    -> Mova levemente a mão durante a gravação para variar os ângulos.")
print(" 3. Pressione 'ESC' quando terminar para treinar o modelo.")
print("="*60 + "\n")

gravando_classe = None
contador_rajada = 0
TAMANHO_RAJADA = 50  # Quantos frames captura automaticamente a cada tecla pressionada

# ==============================================================================
# 2. LAÇO DE CAPTURA PELA WEBCAM
# ==============================================================================
while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        break

    # Espelha horizontalmente (idêntico ao script de teste)
    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resultado = hands.process(frame_rgb)

    coordenadas_atuais = None

    if resultado.multi_hand_landmarks:
        hand_landmarks = resultado.multi_hand_landmarks[0]
        
        mp_drawing.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style()
        )

        # Normalização idêntica ao teste (Ponto 0 / Pulso como origem)
        pulso_x = hand_landmarks.landmark[0].x
        pulso_y = hand_landmarks.landmark[0].y
        pulso_z = hand_landmarks.landmark[0].z

        coordenadas_atuais = []
        for lm in hand_landmarks.landmark:
            coordenadas_atuais.extend([lm.x - pulso_x, lm.y - pulso_y, lm.z - pulso_z])

        # Se estiver no meio de uma gravação em rajada, salva a amostra
        if gravando_classe is not None and contador_rajada > 0:
            dados_coletados.append(coordenadas_atuais)
            labels_coletados.append(gravando_classe)
            contador_rajada -= 1
            if contador_rajada == 0:
                print(f"Gravação da letra '{gravando_classe}' concluída! Total acumulado: {len(labels_coletados)} amostras.")
                gravando_classe = None

    # Interface visual no topo da tela
    cv2.rectangle(frame, (10, 10), (630, 95), (30, 30, 30), -1)
    if gravando_classe is not None:
        status_txt = f"GRAVANDO '{gravando_classe}'... Faltam {contador_rajada} frames"
        cor = (0, 165, 255) # Laranja
    else:
        status_txt = "Pressione uma tecla (A-Z) para gravar | ESC = Treinar"
        cor = (0, 255, 0) # Verde

    cv2.putText(frame, status_txt, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.65, cor, 2)
    cv2.putText(frame, f"Total salvo: {len(labels_coletados)} amostras | Classes: { sorted(list(set(labels_coletados))) }", 
                (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    cv2.imshow("Coletor de Libras - MediaPipe", frame)

    tecla = cv2.waitKey(1) & 0xFF
    if tecla == 27:  # Tecla ESC para encerrar e treinar
        break
    elif (ord('a') <= tecla <= ord('z')) or (ord('A') <= tecla <= ord('Z')) or (ord('0') <= tecla <= ord('9')):
        if coordenadas_atuais is not None and gravando_classe is None:
            gravando_classe = chr(tecla).upper()
            contador_rajada = TAMANHO_RAJADA
            print(f"Iniciando captura de {TAMANHO_RAJADA} amostras para a classe '{gravando_classe}'...")
        elif coordenadas_atuais is None:
            print("Aviso: Mão não detectada na tela! Posicione a mão antes de apertar a tecla.")

cap.release()
cv2.destroyAllWindows()
hands.close()

# ==============================================================================
# 3. SALVAMENTO DO CSV E TREINAMENTO DO RANDOM FOREST
# ==============================================================================
classes_unicas = set(labels_coletados)
if len(classes_unicas) < 2:
    print("\n[Atenção] Grave pelo menos 2 letras diferentes para poder treinar o classificador!")
else:
    df = pd.DataFrame(dados_coletados, columns=colunas)
    df['classe'] = labels_coletados
    df.to_csv(arquivo_csv, index=False)
    print(f"\nDataset atualizado e salvo em '{arquivo_csv}' com {len(df)} amostras.")

    X = df.drop(columns=['classe'])
    y = df['classe'].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print("Treinando o Random Forest com os seus sinais...")
    clf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    acuracia = accuracy_score(y_test, y_pred)

    print("\n" + "="*50)
    print(f"ACURÁCIA ALCANÇADA NO SEU DATASET: {acuracia * 100:.2f}%")
    print("="*50)
    print(classification_report(y_test, y_pred))

    # Salva no formato exato que o script 'testar_asl_webcam.py' já utiliza
    dados_modelo = {
        "modelo": clf,
        "colunas": list(X.columns),
        "num_features": 63
    }
    joblib.dump(dados_modelo, "modelo_asl_rf.pkl")
    print("Modelo salvo com sucesso em 'modelo_asl_rf.pkl'! Agora é só rodar o script de teste da webcam.")


 INSTRUÇÕES DE GRAVAÇÃO PELA WEBCAM:
 1. Faça o sinal com a mão na frente da câmera.
 2. Pressione a tecla da letra correspondente (ex: 'a', 'b', 'c'...).
    -> Cada clique grava uma rajada automática de 50 amostras!
    -> Mova levemente a mão durante a gravação para variar os ângulos.
 3. Pressione 'ESC' quando terminar para treinar o modelo.

Iniciando captura de 50 amostras para a classe 'A'...
Gravação da letra 'A' concluída! Total acumulado: 50 amostras.
Iniciando captura de 50 amostras para a classe 'B'...
Gravação da letra 'B' concluída! Total acumulado: 100 amostras.
Iniciando captura de 50 amostras para a classe 'C'...
Gravação da letra 'C' concluída! Total acumulado: 150 amostras.
Iniciando captura de 50 amostras para a classe 'D'...
Gravação da letra 'D' concluída! Total acumulado: 200 amostras.
Iniciando captura de 50 amostras para a classe 'E'...
Gravação da letra 'E' concluída! Total acumulado: 250 amostras.
Iniciando captura de 50 amostras para a classe 'F'...
Gravaçã